In [ ]:
import os

from PIL import Image
from PIL import ImageDraw
# Import necessary libraries
from transformers import pipeline

# Build the object-detection pipeline using 🤗 Transformers Library
od_pipe = pipeline(task="object-detection", model="facebook/detr-resnet-50")


In [ ]:
# Define helper functions
def load_image_from_path(image_path):
    img = Image.open(image_path)
    return img


def render_results_in_image(image, results):
    draw = ImageDraw.Draw(image)
    for result in results:
        box = result['box']
        label = result['label']
        score = result['score']
        draw.rectangle([(box['xmin'], box['ymin']), (box['xmax'], box['ymax'])], outline="red", width=3)
        draw.text((box['xmin'], box['ymin']), f"{label} ({score:.2f})", fill="red")
    return image


# Load Image
image_path = "img_test_2.jpg"
raw_image = load_image_from_path(image_path)

# Resize image
raw_image = raw_image.resize((600, 400))

# Detect objects in the image
pipeline_output = od_pipe(raw_image)

# Render results on the image
propossed_image = render_results_in_image(raw_image.copy(), pipeline_output)


# Save cropped objects
def crop_and_save_objects(image, pipeline_output, save_dir="cropped_objects"):
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    for i, result in enumerate(pipeline_output):
        box = result['box']
        label = result['label']
        cropped_image = image.crop((box['xmin'], box['ymin'], box['xmax'], box['ymax']))
        path_save = os.path.join(save_dir, f"{label}_{i}.png")
        cropped_image.save(path_save)

crop_and_save_objects(raw_image, pipeline_output)